# 213 — Cluster ranking + iterative reassignment

Walks every run in `outputs/clustering/index.json`, dissolves clusters that
don't bridge BERN (`EL*`) ↔ GVA (`PAT*`) cohorts (i.e. clusters that look
like single-site artifacts) by reassigning their members to the valid
cluster with the highest mean consensus similarity, and recomputes every
downstream metric (consensus matrix, Jaccard, anatomy, condition selectivity,
spatial compactness, centroids) from scratch. Iterates until all surviving
clusters are valid.

Then computes a per-cluster ranking with four weighted axes:

    composite = 4·cross_patient + 3·stability + 2·condition + 1·anatomy
    (each axis min-max normalized across clusters within run, composite
    re-normalized 0–1; rank 1 = best)

**Anatomy axis = mirrored-XYZ spatial compactness** (not aparc purity).
For each cluster: all electrode coords (x, y, z in fsaverage / MNI mm)
are mirrored to one hemisphere (default left, x_new = -|x|) so bilateral
homotopic contacts collapse onto each other; then mean pairwise Euclidean
distance is computed across members. Lower distance = more spatially
compact. The raw anatomy score is the NEGATED distance so min-max
normalization assigns 1.0 to the most compact cluster. Configurable via
`SPATIAL_MIRROR_TO` and `SPATIAL_METRIC` in cell §1. The previous
aparc-purity / entropy score is still computed and saved (as descriptive
metadata in per_cluster_anatomy.csv + inside each ranking entry's
`anatomy_descriptive` block) but no longer drives the axis.

**Per-run outputs** (under each run's run_dir):
* `cluster_ranking.json` — full per-run summary + per-cluster ranks
* `consensus_matrix.npy` (final, post-reassignment)
* `consensus_heatmap.png` (final)
* `per_cluster_stability.csv` + `stability_summary.json` (final)
* `per_cluster_anatomy.csv` + `per_cluster_anatomy.json` (final, descriptive)
* `per_cluster_spatial_compactness.csv` + `.json` (final, drives anatomy axis)
* `cluster_centroids/ranked/cluster_{NN}.png` (square 2×2 figs, bwr)
* `labels.csv` gains a new `cluster_{method}_{feature_set}_ranked` column
  (the original `cluster_{method}_{feature_set}` column is never modified)

**Top-level outputs**:
* `outputs/clustering/index.json` — each processed run gets `has_ranking: true`
* `outputs/clustering/comparisons/{runA_id}_vs_{runB_id}_confusion.{png,csv}`
  for any cross-feature-set comparison run

**Constants:**
* `N_RUNS = 50` KMeans repetitions per consensus matrix (Monti et al. 2003)
* `SUBSAMPLE_FRAC = 0.80` (samples drawn per run without replacement)
* Weights — `cross_patient × 4, stability × 3, condition × 2, anatomy × 1`
* Spatial — `mirror_to=left`, `metric=mean_pairwise`

**References:**
* Monti et al. 2003 — consensus clustering
* Hennig 2007 — per-cluster Jaccard stability
* Hubert & Arabie 1985 — adjusted Rand index (for cross-run comparison)

## 0 — Imports + paths

In [15]:
import os, sys, json, datetime as _dt
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from collections import Counter
from sklearn.metrics import adjusted_rand_score, silhouette_score, silhouette_samples

sys.path.insert(0, str(Path('..').resolve()))

from functions.lf_stability import (
    compute_consensus_matrix,
    per_cluster_jaccard,
    reassign_by_consensus_similarity,
    cluster_center_coverage,
)
from functions.lf_anatomy import (
    cluster_anatomy_purity,
    save_anatomy_artifacts,
    cluster_spatial_compactness,
    save_spatial_compactness_artifacts,
)
from functions import lf_cluster_run as R

CLUSTERING_DIR = Path(R.DEFAULT_OUTPUTS_ROOT)
INDEX_PATH = CLUSTERING_DIR / 'index.json'
APARC_CACHE = CLUSTERING_DIR.parent / '250_recon' / 'fsaverage' / 'aparc_lookup.csv'
COMPARISONS_DIR = CLUSTERING_DIR / 'comparisons'
COMPARISONS_DIR.mkdir(parents=True, exist_ok=True)

assert INDEX_PATH.exists(), f'No {INDEX_PATH} — run a clustering notebook first.'
with open(INDEX_PATH) as f:
    INDEX = json.load(f)
print(f'Found {len(INDEX["runs"])} runs in index.json')
for r in INDEX['runs']:
    print(f"  {r['method']}/{r['feature_set']}/{r['run_id']}  K={r['n_clusters']}  sil={r['silhouette']:.3f}")

Found 63 runs in index.json
  hierarchical/blob/20260521_163124  K=20  sil=0.127
  hierarchical/blob/20260522_205440  K=20  sil=0.127
  hierarchical/blob/20260522_213636  K=20  sil=0.127
  hierarchical/blob/20260523_102742  K=18  sil=0.096
  hierarchical/blob/20260523_104436  K=18  sil=0.096
  hierarchical/blob/20260603_230501  K=18  sil=0.096
  hierarchical/hg/20260521_160447  K=12  sil=0.059
  hierarchical/hg/20260522_205603  K=12  sil=0.059
  hierarchical/hg/20260523_110707  K=20  sil=0.092
  hierarchical/hg/20260528_182149  K=5  sil=0.159
  hierarchical/hg/20260529_185811  K=5  sil=0.159
  hierarchical/hg/20260529_192435  K=5  sil=0.159
  hierarchical/hg/20260530_151854  K=5  sil=0.159
  hierarchical/hg/20260602_101606  K=5  sil=0.159
  hierarchical/hg/20260603_230040  K=5  sil=0.159
  hierarchical/hg/20260604_005708  K=8  sil=0.117
  hierarchical/minus101/20260521_164647  K=10  sil=0.199
  hierarchical/minus101/20260522_210327  K=10  sil=0.199
  hierarchical/minus101/20260523_1043

## 1 — Config

In [16]:
# Consensus matrix
N_RUNS = 50                  # KMeans repetitions (Monti et al. 2003 use 25–50)
SUBSAMPLE_FRAC = 0.80        # fraction of samples per consensus run
RANDOM_STATE = 42

# Iteration safety net (single-pass usually suffices; >1 only if the
# reassignment somehow creates new invalid clusters, which it shouldn't
# because reassignment only ADDS members to already-valid clusters)
MAX_ITERS = 5

# Ranking weights (Step B)
WEIGHT_CROSS_PATIENT = 4
WEIGHT_STABILITY      = 3
WEIGHT_CONDITION      = 2
WEIGHT_ANATOMY        = 1

# Anatomy permutation budget (0 = skip; the existing 211 cell uses 500)
N_PERM_ANATOMY = 0

# Spatial-compactness config — the new "anatomy" axis of the ranking.
# Electrodes are mirrored to one hemisphere so bilateral homotopic
# contacts collapse onto each other (sensible because the language /
# sensorimotor / memory networks are roughly mirror-symmetric).
SPATIAL_MIRROR_TO = 'left'        # 'left' (x = -|x|), 'right' (x = |x|), or 'none'
SPATIAL_METRIC    = 'mean_pairwise'  # 'mean_pairwise' | 'median_pairwise'
                                     # | 'max_pairwise' | 'mean_centroid'

# Which feature sets to process. Add/remove as needed.
FEATURE_SETS = ['hg',  'rawds'] # 'blob', 'minus101', 'raw'

# Skip runs whose cluster_ranking.json already exists?
SKIP_EXISTING = True

# Cohort prefixes
BERN_PREFIX = 'EL'
GVA_PREFIX  = 'PAT'

# Conditions (used for the condition-selectivity score)
CONDITION_COL = 'condition'

# 2D shape used for rendering per-cluster centroid PNGs, per feature set.
# (n_freq, n_time). For 1D feature sets (blob / hg) we render as a single
# row so the 2×2 figure shows a horizontal stripe.
CENTROID_SHAPE = {
    'raw':      (129, 300),
    'minus101': (129, 300),
    'rawds':    (15, 30),
    # blob / hg: shape derived at runtime as (1, n_features)
}

print('Config:')
print(f'  N_RUNS={N_RUNS}  SUBSAMPLE_FRAC={SUBSAMPLE_FRAC}  RANDOM_STATE={RANDOM_STATE}')
print(f'  weights: cross={WEIGHT_CROSS_PATIENT} stab={WEIGHT_STABILITY} '
      f'cond={WEIGHT_CONDITION} anat={WEIGHT_ANATOMY}')
print(f'  spatial: mirror_to={SPATIAL_MIRROR_TO}  metric={SPATIAL_METRIC}')
print(f'  FEATURE_SETS={FEATURE_SETS}  SKIP_EXISTING={SKIP_EXISTING}')

Config:
  N_RUNS=50  SUBSAMPLE_FRAC=0.8  RANDOM_STATE=42
  weights: cross=4 stab=3 cond=2 anat=1
  spatial: mirror_to=left  metric=mean_pairwise
  FEATURE_SETS=['hg', 'rawds']  SKIP_EXISTING=True


## 2 — Anatomical lookup cache (shared across runs)

In [17]:
if not APARC_CACHE.exists():
    raise FileNotFoundError(
        f'aparc cache missing: {APARC_CACHE}\n'
        'Run 211_validation.ipynb (Section C: aparc cache) first to build it.'
    )
df_aparc = pd.read_csv(APARC_CACHE)
print(f'aparc cache: {len(df_aparc)} electrodes, {df_aparc["aparc_label"].nunique()} unique labels')

aparc cache: 3953 electrodes, 34 unique labels


## 3 — Helpers

All the per-run machinery lives in functions defined here so the main loop
below is just one call per run. Nothing here writes to disk except inside
`process_run`, and all paths are explicit (no globbing).

In [18]:
def _reshape_centroid(mean_vec, feature_set):
    """Reshape a flat cluster-mean vector to the natural 2D image for plotting.
    HG is handled separately (line plot)."""
    mean_vec = np.asarray(mean_vec).ravel()
    shape = CENTROID_SHAPE.get(feature_set)
    if shape is None or shape[0] * shape[1] != mean_vec.size:
        # 1D feature set (blob / hg) OR dim mismatch — render as a row
        return mean_vec.reshape(1, -1)
    return mean_vec.reshape(shape)


# Fixed y-limits for the HG centroid line plot. Matches cfg.hg_vmin /
# cfg.hg_vmax (±6.5 dB) so every HG centroid + sample chip shares the same
# y-axis scale as the other ERSP feature sets. The previous dynamic
# ymax-based scaling made every HG centroid look "full-range" regardless
# of how peaky it actually was, which masked across-cluster amplitude
# differences.
HG_YLIM = (-6.5, 6.5)


def _save_centroid_pngs(run_dir, X, labels, feature_set, *, vlim=None):
    """
    Render one PNG per cluster at cluster_centroids/ranked/cluster_{NN}.png
    using figsize=(2, 2).

    Rendering style depends on the feature_set:
      * 'hg'   — time-series line plot of mean high-gamma amplitude.
                 Black mean line + fill_between (red = positive, blue =
                 negative) + faint zero baseline, FIXED ±6.5 dB y-limits
                 (matches cfg.hg_vmin / cfg.hg_vmax + the ERSP cmap scale).
      * other  — square 2x2 figure heatmap with aspect='auto', cmap='bwr'.
                 Matches 212's rawds convention.

    IMPORTANT: the output path is `cluster_centroids/ranked/` (NOT
    `cluster_centroids/k_{K_final}/`). The `k_{K}` namespace is reserved
    for fresh KMeans/HC cuts at K=K (212's backfill for rawds writes
    there). Writing 213's reassigned centroids to `k_{K_final}/` collided
    with that convention and caused MOBA's chip thumbnails to show 213's
    centroids at the K-slider position matching K_final while the labels
    at that slider position came from `cluster_labels_by_k.csv['k_K_final']`
    — two different clusterings, same path. Fixed by writing to the
    distinct `ranked/` subdir.

    `labels` must align row-wise with `X`. Cluster IDs that don't appear in
    `labels` are skipped (e.g. dissolved IDs).
    """
    out_dir = Path(run_dir) / 'cluster_centroids' / 'ranked'
    out_dir.mkdir(parents=True, exist_ok=True)

    uniq = sorted(int(c) for c in np.unique(labels))
    if vlim is None:
        vlim = float(np.percentile(np.abs(X), 99)) or 1.0

    n_written = 0
    for c in uniq:
        idx = np.where(labels == c)[0]
        if idx.size == 0:
            continue
        mean_vec = np.asarray(X[idx].mean(axis=0)).ravel()
        fig, ax = plt.subplots(figsize=(2, 2))

        if feature_set == 'hg':
            # HG = 1D high-gamma envelope. Time-series plot, not an image.
            # Black mean line over red (+) / blue (-) fills, FIXED ±6.5 dB
            # y-limits (HG_YLIM above) so amplitude scale is comparable
            # across all HG centroids + samples.
            t = np.arange(mean_vec.size)
            ax.fill_between(t, 0, mean_vec, where=(mean_vec > 0),
                            color='#d65a5a', alpha=0.55, interpolate=True, linewidth=0)
            ax.fill_between(t, 0, mean_vec, where=(mean_vec < 0),
                            color='#5a7ed6', alpha=0.55, interpolate=True, linewidth=0)
            ax.plot(t, mean_vec, color='black', lw=1.0)
            ax.axhline(0, color='#999', lw=0.35, alpha=0.7)
            ax.set_ylim(*HG_YLIM)
            ax.set_xlim(0, mean_vec.size - 1)
        else:
            img = _reshape_centroid(mean_vec, feature_set)
            ax.imshow(img, aspect='auto', cmap='bwr',
                      vmin=-vlim, vmax=+vlim, interpolation='nearest')

        ax.set_xticks([]); ax.set_yticks([])
        for s in ax.spines.values():
            s.set_visible(False)
        fig.subplots_adjust(left=0, right=1, top=1, bottom=0)
        fig.savefig(out_dir / f'cluster_{c:02d}.png',
                    dpi=100, bbox_inches='tight', pad_inches=0)
        plt.close(fig)
        n_written += 1
    return out_dir, n_written

In [19]:
def _save_consensus_heatmap(run_dir, M, labels, *, n_runs, subsample_frac):
    """Render consensus_heatmap.png reordered by cluster (parallel to lf_stability.save_consensus_artifacts)."""
    labels = np.asarray(labels)
    order = np.argsort(labels, kind='stable')
    M_ord = M[np.ix_(order, order)]
    sorted_labels = labels[order]

    k = int(len(np.unique(labels)))
    fig, ax = plt.subplots(figsize=(7, 6))
    im = ax.imshow(M_ord, cmap='magma', aspect='auto', vmin=0, vmax=1, interpolation='nearest')
    ax.set_title(f'Consensus matrix (K={k}, n_runs={n_runs}, subsample={subsample_frac:.2f})\n'
                 f'diagonal blocks = stable clusters')
    ax.set_xlabel('sample (reordered by cluster)')
    ax.set_ylabel('sample (reordered by cluster)')
    boundaries = np.where(np.diff(sorted_labels) != 0)[0] + 1
    for b in boundaries:
        ax.axhline(b - 0.5, color='white', lw=0.4, alpha=0.6)
        ax.axvline(b - 0.5, color='white', lw=0.4, alpha=0.6)
    plt.colorbar(im, ax=ax, fraction=0.046, label='co-clustering frequency')
    fig.tight_layout()
    fig.savefig(Path(run_dir) / 'consensus_heatmap.png', dpi=120, bbox_inches='tight')
    plt.close(fig)


def _write_stability_artifacts(run_dir, M, labels, *, n_runs, subsample_frac):
    """Write consensus_matrix.npy, per_cluster_stability.csv, stability_summary.json,
    consensus_heatmap.png — mirrors lf_stability.save_consensus_artifacts but for
    a pre-computed consensus matrix (we already have one from the final iteration).
    """
    run_dir = Path(run_dir)
    labels = np.asarray(labels).astype(np.int32)
    np.save(run_dir / 'consensus_matrix.npy', M.astype(np.float32))

    jacc = per_cluster_jaccard(labels, M)
    rows = []
    for c in sorted(jacc.keys()):
        idx = np.where(labels == c)[0]
        rows.append({
            'cluster_id': int(c),
            'size': int(idx.size),
            'jaccard_stability': float(jacc[c]),
            'n_runs': int(n_runs),
        })
    df_stab = pd.DataFrame(rows)
    df_stab.to_csv(run_dir / 'per_cluster_stability.csv', index=False)

    _save_consensus_heatmap(run_dir, M, labels, n_runs=n_runs, subsample_frac=subsample_frac)

    finite = [v for v in jacc.values() if not np.isnan(v)]
    summary = {
        'n_runs': int(n_runs),
        'k': int(len(np.unique(labels))),
        'n_samples': int(len(labels)),
        'subsample_frac': float(subsample_frac),
        'mean_jaccard': float(np.nanmean(list(jacc.values()))) if finite else float('nan'),
        'min_jaccard': float(min(finite)) if finite else float('nan'),
        'max_jaccard': float(max(finite)) if finite else float('nan'),
        'per_cluster_jaccard': {str(c): jacc[c] for c in sorted(jacc.keys())},
    }
    (run_dir / 'stability_summary.json').write_text(json.dumps(summary, indent=2))
    return df_stab, summary, jacc

In [20]:
def _per_cluster_condition_props(labels, df_meta, *, condition_col=CONDITION_COL):
    """
    For each cluster, return {cluster_id: {condition: proportion, ..., max_proportion: float, n_conditions: int}}.

    Used by the Step-B condition selectivity score.
    """
    if condition_col not in df_meta.columns:
        return {int(c): {'max_proportion': float('nan'), 'n_conditions': 0, 'proportions': {}}
                for c in np.unique(labels)}
    conds = df_meta[condition_col].astype(str).to_numpy()
    out = {}
    all_conditions = sorted(set(conds.tolist()))
    for c in np.unique(labels):
        idx = np.where(labels == c)[0]
        if idx.size == 0:
            continue
        cnt = Counter(conds[idx].tolist())
        total = sum(cnt.values())
        props = {k: cnt.get(k, 0) / total for k in all_conditions}
        out[int(c)] = {
            'proportions': props,
            'max_proportion': float(max(props.values())) if props else float('nan'),
            'n_conditions': len(all_conditions),
        }
    return out

In [21]:
def _minmax_norm(values):
    """Min-max normalize a list of floats to [0, 1]. NaNs preserved."""
    a = np.array(values, dtype=np.float64)
    finite = a[np.isfinite(a)]
    if finite.size == 0:
        return [float('nan')] * len(a)
    lo, hi = float(finite.min()), float(finite.max())
    if hi - lo < 1e-12:
        # All equal -> map to 0.5 (so weighting still works)
        return [0.5 if np.isfinite(v) else float('nan') for v in a]
    return [(v - lo) / (hi - lo) if np.isfinite(v) else float('nan') for v in a]


def _compute_ranking(
    cluster_ids,
    *,
    jacc,                  # cluster_id -> jaccard
    coverage,              # cluster_id -> {n_patients, n_bern, n_gva, ...}
    anatomy,               # cluster_id -> {purity, top_3, ...}  (descriptive only)
    spatial,               # cluster_id -> {distance_mm, n_with_coords, ...} — drives the anatomy axis
    condition,             # cluster_id -> {max_proportion, n_conditions, ...}
    total_unique_patients, # int (run-level denominator for cross-patient)
    weights,               # dict with keys cross_patient / stability / condition / anatomy
):
    """
    Build per-cluster raw + normalized + composite scores.

    Axes:
      * cross_patient: (n_patients / total_patients) × (n_centers / 2)
      * stability:     per-cluster Jaccard (Monti consensus)
      * condition:     max(condition_proportion) − 1/n_conditions, clipped [0,1]
      * anatomy:       NEGATED mirrored-XYZ spatial distance (mm) → min-max norm
                       puts the most spatially compact cluster at 1.0.
                       NB: the previous version of this notebook used
                       Desikan-Killiany aparc purity here. Spatial distance
                       is a sharper proxy for "anatomically related"; aparc
                       purity is now stored only as descriptive metadata
                       inside the ranking entry.

    Returns a list of per-cluster entries, each with rank + composite_score
    + raw + normalized substructure, sorted later by rank.
    """
    raw_cross  = []
    raw_stab   = []
    raw_cond   = []
    raw_anat   = []
    cluster_ids = [int(c) for c in cluster_ids]
    for c in cluster_ids:
        cov = coverage.get(c, {})
        n_pat   = cov.get('n_patients', 0)
        n_centers = int(cov.get('has_bern', False)) + int(cov.get('has_gva', False))
        cross = (n_pat / total_unique_patients) * (n_centers / 2.0) if total_unique_patients else 0.0
        raw_cross.append(float(cross))

        raw_stab.append(float(jacc.get(c, float('nan'))))

        condition_entry = condition.get(c, {})
        n_conds = max(1, int(condition_entry.get('n_conditions', 0)))
        max_p   = float(condition_entry.get('max_proportion', float('nan')))
        cond_raw = max_p - (1.0 / n_conds) if np.isfinite(max_p) else float('nan')
        if np.isfinite(cond_raw):
            cond_raw = float(min(1.0, max(0.0, cond_raw)))
        raw_cond.append(cond_raw)

        # Anatomy axis = mirrored-XYZ spatial compactness.
        # Lower distance is better, so we NEGATE before min-max so the
        # most compact cluster gets the highest normalized score.
        spat_entry = spatial.get(c, {})
        dist_mm = spat_entry.get('distance_mm', float('nan'))
        try:
            dist_mm = float(dist_mm)
        except (TypeError, ValueError):
            dist_mm = float('nan')
        raw_anat.append(-dist_mm if np.isfinite(dist_mm) else float('nan'))

    norm_cross = _minmax_norm(raw_cross)
    norm_stab  = _minmax_norm(raw_stab)
    norm_cond  = _minmax_norm(raw_cond)
    norm_anat  = _minmax_norm(raw_anat)

    w_sum_max = (weights['cross_patient'] + weights['stability']
                 + weights['condition'] + weights['anatomy'])

    composite_raw = []
    for i in range(len(cluster_ids)):
        terms = []
        for w, v in [
            (weights['cross_patient'], norm_cross[i]),
            (weights['stability'],     norm_stab[i]),
            (weights['condition'],     norm_cond[i]),
            (weights['anatomy'],       norm_anat[i]),
        ]:
            if np.isfinite(v):
                terms.append(w * v)
        composite_raw.append(sum(terms) / w_sum_max if terms else float('nan'))

    composite_norm = _minmax_norm(composite_raw)
    order = sorted(
        range(len(cluster_ids)),
        key=lambda i: (-composite_norm[i] if np.isfinite(composite_norm[i]) else float('inf')),
    )
    ranks = [0] * len(cluster_ids)
    for r, i in enumerate(order, start=1):
        ranks[i] = r

    entries = []
    for i, c in enumerate(cluster_ids):
        spat_entry = spatial.get(c, {})
        anat_entry = anatomy.get(c, {})
        entries.append({
            'cluster_id': int(c),
            'rank': int(ranks[i]),
            'composite_score': float(composite_norm[i]) if np.isfinite(composite_norm[i]) else None,
            'composite_raw':   float(composite_raw[i])  if np.isfinite(composite_raw[i])  else None,
            'raw': {
                'cross_patient': float(raw_cross[i]) if np.isfinite(raw_cross[i]) else None,
                'stability':     float(raw_stab[i])  if np.isfinite(raw_stab[i])  else None,
                'condition':     float(raw_cond[i])  if np.isfinite(raw_cond[i])  else None,
                # raw anatomy is the NEGATED distance for sign consistency
                # with the other 'higher = better' raw axes; the actual
                # distance lives in spatial.distance_mm below for clarity.
                'anatomy':       float(raw_anat[i])  if np.isfinite(raw_anat[i])  else None,
            },
            'normalized': {
                'cross_patient': float(norm_cross[i]) if np.isfinite(norm_cross[i]) else None,
                'stability':     float(norm_stab[i])  if np.isfinite(norm_stab[i])  else None,
                'condition':     float(norm_cond[i])  if np.isfinite(norm_cond[i])  else None,
                'anatomy':       float(norm_anat[i])  if np.isfinite(norm_anat[i])  else None,
            },
            # Attach the underlying spatial + anatomy entries so MOBA + the
            # PDF export can show distance + region info without joining
            # multiple CSVs.
            'spatial': {
                'distance_mm':   (float(spat_entry.get('distance_mm'))
                                  if np.isfinite(spat_entry.get('distance_mm', float('nan'))) else None),
                'n_total':        int(spat_entry.get('n_total', 0)),
                'n_with_coords':  int(spat_entry.get('n_with_coords', 0)),
                'metric':         spat_entry.get('metric'),
                'mirror_to':      spat_entry.get('mirror_to'),
                'centroid_xyz':   spat_entry.get('centroid_xyz'),
            },
            'anatomy_descriptive': {
                'purity':         (float(anat_entry.get('purity'))
                                   if np.isfinite(anat_entry.get('purity', float('nan'))) else None),
                'entropy_bits':   (float(anat_entry.get('entropy_bits'))
                                   if np.isfinite(anat_entry.get('entropy_bits', float('nan'))) else None),
                'top_3':          anat_entry.get('top_3', []),
            },
        })
    return entries

In [22]:
def process_run(
    run,
    *,
    n_runs=N_RUNS,
    subsample_frac=SUBSAMPLE_FRAC,
    random_state=RANDOM_STATE,
    weights=None,
    n_perm_anatomy=N_PERM_ANATOMY,
    spatial_mirror_to=SPATIAL_MIRROR_TO,
    spatial_metric=SPATIAL_METRIC,
    max_iters=MAX_ITERS,
    bern_prefix=BERN_PREFIX,
    gva_prefix=GVA_PREFIX,
    verbose=True,
):
    """
    Process one run from index.json end-to-end:
      1. Load X_train.npy + labels.csv
      2. Iteratively dissolve clusters that don't bridge BERN ↔ GVA
      3. Recompute consensus / Jaccard / anatomy / condition / spatial
         compactness / centroids on the final labels
      4. Compute the 4-axis ranking (anatomy axis = NEGATED mirrored-XYZ
         distance, so most compact cluster wins)
      5. Write all artifacts + return the ranking dict

    Returns the per-run ranking summary (also serialized to cluster_ranking.json).
    """
    weights = weights or {
        'cross_patient': WEIGHT_CROSS_PATIENT,
        'stability':      WEIGHT_STABILITY,
        'condition':      WEIGHT_CONDITION,
        'anatomy':        WEIGHT_ANATOMY,
    }

    method      = run['method']
    feature_set = run['feature_set']
    run_dir     = CLUSTERING_DIR / run['path']
    cluster_col = f"cluster_{method}_{feature_set}"

    X_path     = run_dir / 'X_train.npy'
    labels_csv = run_dir / 'labels.csv'
    if not X_path.exists():
        return {'skipped': True, 'reason': f'X_train.npy missing: {X_path}'}
    if not labels_csv.exists():
        return {'skipped': True, 'reason': f'labels.csv missing: {labels_csv}'}

    X = np.load(X_path)
    df_meta = pd.read_csv(labels_csv)
    if cluster_col not in df_meta.columns:
        cands = [c for c in df_meta.columns if c.startswith('cluster_') and not c.endswith('_ranked')]
        if not cands:
            return {'skipped': True, 'reason': f'no cluster_* column in {labels_csv}'}
        cluster_col = cands[0]
    if 'patient_id' not in df_meta.columns:
        return {'skipped': True, 'reason': f'labels.csv missing patient_id'}
    if len(df_meta) != X.shape[0]:
        return {'skipped': True, 'reason':
                f'labels.csv ({len(df_meta)}) vs X_train ({X.shape[0]}) mismatch'}

    labels = df_meta[cluster_col].to_numpy().astype(np.int32)
    K_original = int(len(np.unique(labels)))
    K = K_original

    if verbose:
        print(f'  loaded X={X.shape}, labels={labels.shape}, K_original={K_original}')

    dissolved = []
    consensus_matrix = None
    iter_n = 0

    # ------------------------------------------------------------------
    # ITERATIVE REASSIGNMENT (Step A)
    # ------------------------------------------------------------------
    while iter_n < max_iters:
        iter_n += 1
        if verbose:
            print(f'  --- iter {iter_n}: consensus matrix at K={K} ---')
        consensus_matrix = compute_consensus_matrix(
            X, K,
            n_runs=n_runs,
            subsample_frac=subsample_frac,
            random_state=random_state,
            verbose=False,
        )
        coverage = cluster_center_coverage(
            labels, df_meta,
            bern_prefix=bern_prefix, gva_prefix=gva_prefix,
        )
        invalid = [c for c, info in coverage.items() if not info['is_valid']]
        valid   = [c for c, info in coverage.items() if info['is_valid']]

        if verbose:
            print(f'    invalid={invalid}  valid={len(valid)} clusters')

        if not invalid:
            if verbose:
                print(f'    stable (all clusters bridge BERN↔GVA). Stopping.')
            break
        if not valid:
            print(f'  [warn] {run["path"]}: no valid clusters in iter {iter_n}; '
                  f'cannot reassign. Leaving labels as-is.')
            break

        old_labels = labels.copy()
        labels = reassign_by_consensus_similarity(
            old_labels, consensus_matrix, invalid, valid,
        ).astype(np.int32)
        # Record what got merged where
        for ic in invalid:
            members = np.where(old_labels == ic)[0]
            if members.size == 0:
                continue
            target = int(labels[members[0]])
            dissolved.append({
                'iteration':    int(iter_n),
                'from_cluster': int(ic),
                'to_cluster':   target,
                'n_members':    int(members.size),
            })
        K = int(len(np.unique(labels)))

    K_final = int(len(np.unique(labels)))
    if verbose:
        print(f'  iterations={iter_n}  K_original={K_original}  K_final={K_final}  '
              f'dissolved={len(dissolved)}')

    # ------------------------------------------------------------------
    # FINAL METRICS (run on the converged labels)
    # ------------------------------------------------------------------
    # 1. Stability artifacts (consensus_matrix.npy + heatmap + per-cluster CSV + summary)
    df_stab, stab_summary, jacc = _write_stability_artifacts(
        run_dir, consensus_matrix, labels,
        n_runs=n_runs, subsample_frac=subsample_frac,
    )

    # 2. Anatomy artifacts — feed a synthetic df_labels with the FINAL labels
    df_labels_for_anat = df_meta.copy()
    df_labels_for_anat[cluster_col] = labels  # overwrite in-memory only
    df_anat, anat_res = save_anatomy_artifacts(
        run_dir, df_labels_for_anat, df_aparc,
        cluster_col=cluster_col, n_perm=n_perm_anatomy, verbose=False,
    )
    anatomy_by_cluster = anat_res  # cluster_id -> {purity, top_3, entropy_bits, ...}

    # 2b. NEW — Spatial compactness (mirrored-XYZ pairwise distance).
    #     This drives the ranking's "anatomy" axis (replacing purity).
    #     Lower distance = more spatially compact = higher anatomy score.
    df_spat, spat_res = save_spatial_compactness_artifacts(
        run_dir, df_labels_for_anat, df_aparc,
        cluster_col=cluster_col,
        mirror_to=spatial_mirror_to,
        metric=spatial_metric,
        verbose=False,
    )
    spatial_by_cluster = spat_res  # cluster_id -> {distance_mm, centroid_xyz, ...}

    # 3. Condition selectivity
    condition_by_cluster = _per_cluster_condition_props(labels, df_meta)

    # 4. Coverage on final labels
    coverage_final = cluster_center_coverage(
        labels, df_meta, bern_prefix=bern_prefix, gva_prefix=gva_prefix,
    )

    # 5. Per-cluster centroid PNGs (square 2×2 figures) — written to
    #    cluster_centroids/ranked/ (NOT k_{K_final}/) to avoid colliding
    #    with 212's fresh K-cut PNGs at the same K_final value.
    out_dir, n_png = _save_centroid_pngs(run_dir, X, labels, feature_set)
    if verbose:
        print(f'  centroid PNGs: {n_png} -> {out_dir}')

    # 6. Silhouette + (if k_range exists) gap statistic recomputed on final labels.
    #    Silhouette here is overall + per-cluster on the FINAL clustering.
    if len(np.unique(labels)) > 1:
        sil_overall = float(silhouette_score(X, labels))
        sil_per_point = silhouette_samples(X, labels)
    else:
        sil_overall = float('nan'); sil_per_point = np.full(len(labels), float('nan'))
    # Re-fit gap statistic for runs that originally had k_range. We only
    # evaluate gap at K_final (one K, not the whole sweep) to avoid blowing
    # up runtime. The existing gap_by_k.json (when present) stays put.
    manifest_path = run_dir / 'manifest.json'
    manifest_local = json.loads(manifest_path.read_text()) if manifest_path.exists() else {}
    had_k_range = bool(manifest_local.get('summary', {}).get('k_range'))
    gap_entry = None
    if had_k_range:
        try:
            gap_local = R._compute_gap_statistic(
                X, [K_final], n_refs=10, random_state=random_state, verbose=False,
            )
            gap_entry = {str(K_final): gap_local.get(K_final)}
        except Exception as e:
            print(f'  [warn] gap stat at K_final={K_final} failed: {e}')
            gap_entry = None

    # ------------------------------------------------------------------
    # 7. RANKING (Step B) — anatomy axis = spatial compactness
    # ------------------------------------------------------------------
    cluster_ids = sorted(int(c) for c in np.unique(labels))
    total_unique_patients = int(df_meta['patient_id'].astype(str).nunique())
    rank_entries = _compute_ranking(
        cluster_ids,
        jacc=jacc,
        coverage=coverage_final,
        anatomy=anatomy_by_cluster,
        spatial=spatial_by_cluster,
        condition=condition_by_cluster,
        total_unique_patients=total_unique_patients,
        weights=weights,
    )
    # Attach cluster size + condition proportions + top regions onto each entry so
    # MOBA can render the chip badges without joining four CSVs.
    for e in rank_entries:
        c = e['cluster_id']
        idx = np.where(labels == c)[0]
        cov = coverage_final.get(c, {})
        e['size']         = int(idx.size)
        e['n_patients']   = int(cov.get('n_patients', 0))
        e['n_bern']       = int(cov.get('n_bern', 0))
        e['n_gva']        = int(cov.get('n_gva', 0))
        e['has_bern']     = bool(cov.get('has_bern', False))
        e['has_gva']      = bool(cov.get('has_gva', False))
        e['silhouette']   = (float(np.nanmean(sil_per_point[idx]))
                              if idx.size and np.isfinite(sil_overall)
                              else float('nan'))
        e['top_3_regions'] = anatomy_by_cluster.get(c, {}).get('top_3', [])
        e['condition_proportions'] = condition_by_cluster.get(c, {}).get('proportions', {})

    ranking_summary = {
        'run_id':                  run['run_id'],
        'method':                  method,
        'feature_set':             feature_set,
        'n_clusters_original':     int(K_original),
        'n_clusters_final':        int(K_final),
        'reassignment_iterations': int(iter_n),
        'clusters_dissolved':      dissolved,
        'weights':                 dict(weights),
        'consensus': {
            'n_runs':         int(n_runs),
            'subsample_frac': float(subsample_frac),
            'random_state':   int(random_state),
        },
        # Record the spatial config so consumers can interpret the anatomy axis
        'spatial': {
            'mirror_to': spatial_mirror_to,
            'metric':    spatial_metric,
            'note':      'anatomy axis = negated spatial distance (lower mm = higher score after min-max norm)',
        },
        'silhouette_overall_final': float(sil_overall) if np.isfinite(sil_overall) else None,
        'gap_at_k_final':           gap_entry,
        'clusters':                 sorted(rank_entries, key=lambda e: e['rank']),
        'cluster_col_ranked':       f'{cluster_col}_ranked',
        'created_at':              _dt.datetime.utcnow().isoformat(timespec='seconds') + 'Z',
    }
    (run_dir / 'cluster_ranking.json').write_text(json.dumps(ranking_summary, indent=2, default=str))

    # 8. Append _ranked column to labels.csv (preserve the original column verbatim)
    df_labels_disk = pd.read_csv(labels_csv)
    ranked_col = f'{cluster_col}_ranked'
    df_labels_disk[ranked_col] = labels.astype(np.int32)
    df_labels_disk.to_csv(labels_csv, index=False)

    # 9. Flag index.json so MOBA / consumers know this run has ranking artifacts.
    #    (We do this here per-run so partial runs are visible immediately.)
    with open(INDEX_PATH) as f:
        idx_doc = json.load(f)
    for r in idx_doc.get('runs', []):
        if (r.get('method') == method
                and r.get('feature_set') == feature_set
                and r.get('run_id') == run['run_id']):
            r['has_ranking']  = True
            r['n_clusters_ranked'] = K_final
            break
    idx_doc['updated_at'] = _dt.datetime.utcnow().isoformat(timespec='seconds') + 'Z'
    INDEX_PATH.write_text(json.dumps(idx_doc, indent=2, default=str))

    return ranking_summary

## 4 — Main per-run loop

For each run in `INDEX['runs']`: load X + labels, run the iterative
reassignment, write all artifacts, append `_ranked` column to `labels.csv`,
flag `index.json`. Skips runs whose `cluster_ranking.json` already exists
(set `SKIP_EXISTING = False` above to force re-run).

In [26]:
ALL_SUMMARIES = []
for run in INDEX['runs']:
    if run['feature_set'] not in FEATURE_SETS:
        continue
    run_dir = CLUSTERING_DIR / run['path']
    ranking_path = run_dir / 'cluster_ranking.json'
    if SKIP_EXISTING and ranking_path.exists():
        print(f'[skip] {run["path"]} (cluster_ranking.json exists)')
        continue
    print(f'\n=== {run["path"]}  K={run["n_clusters"]} ===')
    try:
        s = process_run(run, verbose=True)
    except Exception as e:
        print(f'  [ERROR] {run["path"]}: {type(e).__name__}: {e}')
        import traceback; traceback.print_exc()
        continue
    if s.get('skipped'):
        print(f'  [skip] {s["reason"]}')
        continue
    ALL_SUMMARIES.append(s)
    print(f'  -> K_final={s["n_clusters_final"]} '
          f'dissolved={len(s["clusters_dissolved"])} '
          f'iters={s["reassignment_iterations"]} '
          f'mean_jaccard={np.mean([c["raw"]["stability"] for c in s["clusters"] if c["raw"]["stability"] is not None]):.3f}')

print(f'\nProcessed {len(ALL_SUMMARIES)} runs.')


=== hierarchical/hg/runs/20260521_160447  K=12 ===
  [skip] X_train.npy missing: \\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_LoraFanda\02_FBM_Clustering\outputs\clustering\hierarchical\hg\runs\20260521_160447\X_train.npy

=== hierarchical/hg/runs/20260522_205603  K=12 ===
  [skip] X_train.npy missing: \\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_LoraFanda\02_FBM_Clustering\outputs\clustering\hierarchical\hg\runs\20260522_205603\X_train.npy

=== hierarchical/hg/runs/20260523_110707  K=20 ===
  [skip] X_train.npy missing: \\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_LoraFanda\02_FBM_Clustering\outputs\clustering\hierarchical\hg\runs\20260523_110707\X_train.npy

=== hierarchical/hg/runs/20260528_182149  K=5 ===
  [skip] X_train.npy missing: \\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_LoraFanda\02_FBM_Clustering\outputs\clustering\hierarchical\hg\runs\20260528_182149\X_train.npy
[skip] hierarchical/hg/runs/20260529_185811 (cluster_

## 5 — Preview: ranking table per processed run

In [27]:
for s in ALL_SUMMARIES:
    print(f'\n--- {s["method"]}/{s["feature_set"]}/{s["run_id"]} '
          f'(K={s["n_clusters_original"]} -> {s["n_clusters_final"]}, '
          f'dissolved={len(s["clusters_dissolved"])}) ---')
    rows = []
    for c in s['clusters']:
        spat = c.get('spatial', {}) or {}
        rows.append({
            'rank':       c['rank'],
            'cluster_id': c['cluster_id'],
            'size':       c['size'],
            'n_pat':      c['n_patients'],
            'centers':    f"{'B' if c['has_bern'] else '-'}{'G' if c['has_gva'] else '-'}",
            'composite':  c['composite_score'],
            'cross_n':    c['normalized']['cross_patient'],
            'stab_n':     c['normalized']['stability'],
            'cond_n':     c['normalized']['condition'],
            'anat_n':     c['normalized']['anatomy'],
            'dist_mm':    spat.get('distance_mm'),
            'top_region': (c['top_3_regions'][0][0] if c['top_3_regions'] else ''),
        })
    df_rank = pd.DataFrame(rows).sort_values('rank')
    print(df_rank.to_string(index=False, float_format=lambda v: f'{v:.3f}' if isinstance(v, (int, float)) and v is not None else ''))


--- hierarchical/hg/20260604_005708 (K=8 -> 8, dissolved=0) ---
 rank  cluster_id  size  n_pat centers  composite  cross_n  stab_n  cond_n  anat_n  dist_mm       top_region
    1           2    82     12      BG      1.000    0.235   0.994   0.763   0.770   39.719           insula
    2           7   137     17      BG      0.919    0.529   1.000   0.423   0.000   61.722           insula
    3           5   657     25      BG      0.553    1.000   0.114   0.103   0.252   54.523           insula
    4           0   255     14      BG      0.367    0.353   0.214   0.863   0.428   49.494   middletemporal
    5           3   562     22      BG      0.210    0.824   0.000   0.000   0.414   49.888   middletemporal
    6           1    88     13      BG      0.190    0.294   0.138   0.528   1.000   33.164   middletemporal
    7           4   261     18      BG      0.189    0.588   0.021   0.518   0.189   56.328   middletemporal
    8           6    53      8      BG      0.000    0.000   0.

## 6 — Cross-feature-set / cross-method comparison (Step D)

`compare_runs(run_a, run_b)` computes the adjusted Rand index between the
`_ranked` columns of two runs and writes a confusion-matrix heatmap +
CSV to `outputs/clustering/comparisons/{runA_id}_vs_{runB_id}_confusion.{png,csv}`.

Rows of the matrix = run A's clusters ordered by ranking score (rank 1
at the top). Columns = run B's clusters ordered by ranking score.

In [28]:
def _load_ranking(run):
    """Return (labels, df_meta, ranking_summary) for a run dict from INDEX."""
    run_dir = CLUSTERING_DIR / run['path']
    df = pd.read_csv(run_dir / 'labels.csv')
    method = run['method']; feature_set = run['feature_set']
    ranked_col = f'cluster_{method}_{feature_set}_ranked'
    if ranked_col not in df.columns:
        raise FileNotFoundError(f'_ranked column missing in {run_dir / "labels.csv"}; '
                                f'run process_run on this run first.')
    ranking_path = run_dir / 'cluster_ranking.json'
    ranking = json.loads(ranking_path.read_text())
    return df[ranked_col].to_numpy(), df, ranking


def compare_runs(run_a, run_b, *, sample_idx_col='sample_idx'):
    """
    Compare two runs by their _ranked columns. Saves PNG + CSV under
    `outputs/clustering/comparisons/{a_id}_vs_{b_id}_confusion.{png,csv}`
    (per user spec — filename is just the run_ids; method + feature_set
    are surfaced on the figure axes so disambiguation is preserved when
    timestamps happen to match across (method, feature_set)).

    Returns (ari, confusion_matrix_dataframe).
    """
    labels_a, df_a, rank_a = _load_ranking(run_a)
    labels_b, df_b, rank_b = _load_ranking(run_b)

    # Align by sample_idx in case row order differs between the two labels.csvs
    if sample_idx_col in df_a.columns and sample_idx_col in df_b.columns:
        merged = pd.merge(
            df_a[[sample_idx_col]].assign(la=labels_a),
            df_b[[sample_idx_col]].assign(lb=labels_b),
            on=sample_idx_col, how='inner',
        )
        la = merged['la'].to_numpy(); lb = merged['lb'].to_numpy()
    else:
        if len(labels_a) != len(labels_b):
            raise ValueError(
                f'labels length mismatch ({len(labels_a)} vs {len(labels_b)}) and '
                f'no sample_idx column to align by'
            )
        la = labels_a; lb = labels_b

    ari = float(adjusted_rand_score(la, lb))

    # Cluster orderings driven by ranking (best at top/left)
    a_order = [int(c['cluster_id']) for c in sorted(rank_a['clusters'], key=lambda x: x['rank'])]
    b_order = [int(c['cluster_id']) for c in sorted(rank_b['clusters'], key=lambda x: x['rank'])]

    mat = np.zeros((len(a_order), len(b_order)), dtype=np.int64)
    a_index = {c: i for i, c in enumerate(a_order)}
    b_index = {c: i for i, c in enumerate(b_order)}
    for ai, bj in zip(la, lb):
        if int(ai) in a_index and int(bj) in b_index:
            mat[a_index[int(ai)], b_index[int(bj)]] += 1

    df_mat = pd.DataFrame(
        mat,
        index=[f'A_rank{i+1}_c{c}' for i, c in enumerate(a_order)],
        columns=[f'B_rank{i+1}_c{c}' for i, c in enumerate(b_order)],
    )

    a_id = run_a['run_id']; b_id = run_b['run_id']
    a_tag = f"{run_a['method']}-{run_a['feature_set']}-{a_id}"
    b_tag = f"{run_b['method']}-{run_b['feature_set']}-{b_id}"
    base = COMPARISONS_DIR / f'{a_id}_vs_{b_id}_confusion'
    df_mat.to_csv(base.with_suffix('.csv'))

    fig, ax = plt.subplots(figsize=(max(5, 0.35 * mat.shape[1] + 2),
                                     max(4, 0.35 * mat.shape[0] + 2)))
    im = ax.imshow(mat, aspect='auto', cmap='magma_r', interpolation='nearest')
    ax.set_xticks(range(mat.shape[1])); ax.set_yticks(range(mat.shape[0]))
    ax.set_xticklabels(df_mat.columns, rotation=90, fontsize=7)
    ax.set_yticklabels(df_mat.index, fontsize=7)
    ax.set_xlabel(f'B: {b_tag}  (cols ordered by rank)')
    ax.set_ylabel(f'A: {a_tag}  (rows ordered by rank)')
    ax.set_title(f'Confusion matrix · ARI = {ari:.3f}')
    # Annotate cells with the count when there are few clusters
    if mat.shape[0] <= 25 and mat.shape[1] <= 25:
        for i in range(mat.shape[0]):
            for j in range(mat.shape[1]):
                if mat[i, j] > 0:
                    ax.text(j, i, str(int(mat[i, j])), ha='center', va='center',
                            color='white' if mat[i, j] > mat.max() * 0.5 else 'black',
                            fontsize=6)
    plt.colorbar(im, ax=ax, fraction=0.046)
    fig.tight_layout()
    fig.savefig(base.with_suffix('.png'), dpi=130, bbox_inches='tight')
    plt.close(fig)

    print(f'compare_runs: ARI={ari:.3f} -> {base}.png + .csv')
    return ari, df_mat

### Example: compare KMeans/rawds vs Hierarchical/rawds (the two latest runs)

Picks the most recent runs of each method × `rawds` feature set. Edit the
selection below for other comparisons (e.g. across feature sets).

In [29]:
def _latest_run(method, feature_set):
    cands = [r for r in INDEX['runs']
             if r['method'] == method and r['feature_set'] == feature_set]
    if not cands:
        return None
    return sorted(cands, key=lambda r: r['run_id'])[-1]

PAIRS_TO_COMPARE = [
    (_latest_run('kmeans', 'rawds'),       _latest_run('hierarchical', 'rawds')),
    (_latest_run('kmeans', 'rawds'),       _latest_run('kmeans', 'hg')),
    (_latest_run('kmeans', 'rawds'),       _latest_run('kmeans', 'blob')),
]

for a, b in PAIRS_TO_COMPARE:
    if a is None or b is None:
        print(f'[skip] missing run in pair (a={a}, b={b})'); continue
    try:
        compare_runs(a, b)
    except FileNotFoundError as e:
        print(f'[skip] {e}')

compare_runs: ARI=0.365 -> \\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_LoraFanda\02_FBM_Clustering\outputs\clustering\comparisons\20260605_112054_vs_20260605_112125_confusion.png + .csv
compare_runs: ARI=0.231 -> \\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_LoraFanda\02_FBM_Clustering\outputs\clustering\comparisons\20260605_112054_vs_20260604_005648_confusion.png + .csv
[skip] _ranked column missing in \\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_LoraFanda\02_FBM_Clustering\outputs\clustering\kmeans\blob\runs\20260603_230453\labels.csv; run process_run on this run first.


## Done

Commit the new artifacts:
```
git add 02_FBM_Clustering/outputs/clustering
git commit -m "213: cluster ranking + iterative center-coverage reassignment"
git push
```

Then load the website to see the ranking-driven sort + score badges + dissolved indicator.